# Suite PYT — The Python Types dialect

Python's type expressions, as the typing specification defines them, and its type parameter lists (the Language
Reference, 8.11): the dialect `Expressions.Python.Types` of the Python family, its kinds mbse-programs' type parameters,
its `Term` listing them beside Python3's branches, which a type expression is, the position of Python3's that holds it,
its holder, what a type parameter binds, its editions, its printing, and its evaluation, which refuses each kind by
name (the dialects proposal, Retrofit's design, Python).

In [ ]:
import functools
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Evaluators as PV, Expressions as P, Text as PText
from mbse.Expressions.Python.Patterns import Expressions as PP
from mbse.Expressions.Python.Types import Expressions as PT, Text as PTText
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the framework, the standard dialects Python3's complex numbers use, and the Python family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
                PythonTypes.install, PythonPatterns.install]:
    store.transaction(install)
store.transaction(PythonTypes.define)  # the versions the Types dialect covers
Term = store.registered("Expressions.Framework.Term")
READER = {d.union: d for d in (P.DIALECT, PT.DIALECT, PP.DIALECT)}  # the dialect whose union holds a term


def held(spec, dialect=PT.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the Types dialect's unless given), written in a
    transaction."""
    with store.transaction():
        holders = functools.reduce(getattr, dialect.path.split("."), store.Expressions)
        return holders.Expression(lambda e: e.root(dialect.spec(spec)))


typed = lambda spec: F.term(held(spec))  # noqa: E731: a term of the Types dialect, held by its holder
python = lambda spec: F.term(held(spec, P.DIALECT))  # noqa: E731: a term of Python3, held by its holder
optional = P.binop("|", P.name("int"), P.constant(None))  # `int | None`, a type expression of Python3's kinds

<a id="PYT-01"></a>
## PYT-01 · The Types dialect's union, its kinds' schemas in mbse-programs' shapes, the position that holds it, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
# Its kinds: mbse-programs' type parameters, named and shaped as it has them; its Term lists them after Python3's
# branches, which a type expression is (an annotation, a bound or a default may be any expression), invalid syntax among
# them; it has no helper of its own, its names str.
union = lambda dialect: [b.name for b in store.registered(dialect.union).branches]  # noqa: E731: its Term's branches
assert PT.TAGS == ("typevar", "paramspec", "typevartuple") and PT.HELPERS == ()
assert union(PT.DIALECT) == [*union(P.DIALECT), *PT.TAGS]  # Python3's installed first
assert union(P.DIALECT)[-2:] == ["error", "missing"] and not set(PT.TAGS) & set(union(P.DIALECT))
assert list(PT.DIALECT.kinds()) == [*PT.TAGS, "error", "missing"]
assert PT.DIALECT.name() == "Python Types" and PT.DIALECT.tag == "python_types"
assert PT.DIALECT.path == "Python.Types" and PT.DIALECT.union == "Expressions.Python.Types.Term"
TypesTerm, Python3Term = store.registered(PT.DIALECT.union), store.registered(P.DIALECT.union)
kinds = {b.name: b.schema for b in TypesTerm.branches}
assert [kinds[tag].name for tag in PT.TAGS] == [f"Expressions.Python.Types.{kind}" for kind in ("TypeVar", "ParamSpec", "TypeVarTuple")]
assert {tag: list(kinds[tag].properties) for tag in PT.TAGS} == {  # each a value object, in mbse-programs' order
    "typevar": ["name", "bound", "default_value"], "paramspec": ["name", "default_value"],
    "typevartuple": ["name", "default_value"]} and not any(kinds[tag].ref for tag in PT.TAGS)
assert kinds["typevar"].properties["bound"].schema is TypesTerm and all(
    kinds[tag].properties["default_value"].schema is TypesTerm for tag in PT.TAGS)  # each a type expression
assert F._parts(kinds["typevar"]) == (("bound", "default_value"), (), ("name",))
assert F._parts(kinds["paramspec"]) == F._parts(kinds["typevartuple"]) == (("default_value",), (), ("name",))
assert all(kinds[b.name] is b.schema for b in Python3Term.branches)  # one schema a branch of both unions ("(b)")
# The position that holds its Term: a parameter's annotation (Python3's `arg`, which mbse-programs' function definitions
# share); its Term is a branch of the framework's, its dialect a singleton.
Arg = store.registered("Expressions.Python.Python3.Arg")
assert Arg.properties["annotation"].schema is TypesTerm and Arg.properties["default_value"].schema is Python3Term
assert {b.name: b.schema for b in Term.branches}["python_types"] is TypesTerm
assert store.singleton("Expressions.Python.Types") is store.Instances.Expressions.Python.Types
# Each dialect of the family reads its kinds, and it reads theirs, where its Term lists them.
assert P.DIALECT.kind_named("typevar") is PT.DIALECT.kinds()["typevar"]
assert PT.DIALECT.kind_named("binop") is P.DIALECT.kinds()["binop"]
assert PT.DIALECT.reads() == (P.DIALECT, PP.DIALECT) and P.DIALECT.reads() == (PT.DIALECT, PP.DIALECT)
# Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
# validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
Holder = store.registered("Expressions.Python.Types.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is TypesTerm and Holder.validate() == [] and list(Holder.adjacencies) == ["abi"]
every = lambda u: u.typevar(lambda t: t.name("T").bound(  # noqa: E731: T: list[int] | None = int
    P.binop("|", P.subscript(P.name("list"), P.name("int")), P.constant(None))).default_value(P.name("int")))
with store.transaction():
    holder = store.Expressions.Python.Types.Expression(lambda e: e.root(PT.DIALECT.spec(every)))
    linked = store.Schemas.OfExpression(lambda s: s.name("pyt.every").expression(lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and PT.DIALECT.validate(holder, {"list"}) == []
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
assert F.dialect_of(F.term(holder)) == PT.DIALECT.union and PTText.ToText(holder) == "T: list[int] | None = int"
for write, read in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(read(Holder, write(Holder, holder)).root, holder.root)
    assert F.same(read(TypesTerm, write(TypesTerm, F.term(holder))), F.term(holder))
# The schema refuses a type parameter where a union that lacks it is held: Python3's holder's root.
with raises(Exception, match="'typevar' is not a member of a union value"):
    with store.transaction():
        store.Expressions.Python.Python3.Expression(lambda e: e.root(PT.DIALECT.make(F.Form("typevar", {"name": "T"}, (None, None)))))
# Another family's term is refused, as before.
with raises(ValueError, match="a Patterns let is not a Python Types term"):
    PT.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT)))

<a id="PYT-02"></a>
## PYT-02 · Each kind and its checks: a name that is an identifier, a bound and a default that are optional; a type expression of Python3's kinds; forms and `make`, and inference

In [ ]:
# Each kind's checks: its name, required, an identifier; its bound and its default optional, each a type expression,
# whose names are references the scope must bind. A type expression is any of Python3's kinds, checked as Python3's.
validate = lambda spec, bound=(): PT.DIALECT.validate(typed(spec), bound)  # noqa: E731
for spec, problems in [
    (lambda u: u.typevar(lambda t: t.name("T")), []),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.name("int")).default_value(P.name("str"))), []),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.tuple_(P.name("int"), P.name("str")))), []),  # constrained
    (lambda u: u.typevar(lambda t: t), ["a typevar needs a name"]),
    (lambda u: u.typevar(lambda t: t.name("1T")), ["a typevar's name must be an identifier, got '1T'"]),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.name("U"))), ["bound: variable 'U' is not bound"]),
    (lambda u: u.paramspec(lambda t: t.name("P")), []),
    (lambda u: u.paramspec(lambda t: t.name("P").default_value(P.list_(P.name("int"), P.name("str")))), []),
    (lambda u: u.paramspec(lambda t: t), ["a paramspec needs a name"]),
    (lambda u: u.paramspec(lambda t: t.name("P Q")), ["a paramspec's name must be an identifier, got 'P Q'"]),
    (lambda u: u.typevartuple(lambda t: t.name("Ts").default_value(P.starred(P.subscript(P.name("tuple"), P.name("int"))))),
     ["default_value: value: value: variable 'tuple' is not bound"]),
    (lambda u: u.typevartuple(lambda t: t.name("")), ["a typevartuple needs a name"]),
    (P.subscript(P.name("dict"), P.tuple_(P.name("str"), P.name("int"))), ["value: variable 'dict' is not bound"]),
    (optional, []),  # a type expression: Python3's kinds, checked as Python3's
    (P.unaryop("not", P.name("int")), []),  # which ones the typing specification allows is a constraint (Decided, 4)
]:
    assert validate(spec) == problems, (problems, validate(spec))
assert validate(lambda u: u.typevar(lambda t: t.name("T").bound(P.name("U"))), {"U"}) == []
# Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
# from which make writes it again, by the dialect whose union holds it; its name is the name the framework reads it by.
assert all(PT.DIALECT.kinds()[tag].role == F.APPLICATION for tag in PT.TAGS)
tree = typed(every)
walked = list(F.walk(tree))
assert [F.kind_of(node)[0] for node in walked] == ["typevar", "binop", "subscript", "variable", "variable", "constant", "variable"]
assert [F.dialect_of(node) for node in walked] == [PT.DIALECT.union] * 2 + [P.DIALECT.union] * 4 + [PT.DIALECT.union]
for node in walked:
    dialect = READER[F.dialect_of(node)]
    made = F.term(held(dialect.make(F.form(node)), dialect))
    assert F.same(made, node) and made is not node
assert F.form(typed(lambda u: u.typevar(lambda t: t.name("T")))) == F.Form("typevar", {"name": "T"}, (None, None))
assert F.form(typed(lambda u: u.paramspec(lambda t: t.name("P")))) == F.Form("paramspec", {"name": "P"}, (None,))
assert F.name_of(tree) == "T" and F.name_of(typed(lambda u: u.typevartuple(lambda t: t.name("Ts")))) == "Ts"
assert PT.DIALECT.infer(typed(lambda u: u.typevar(lambda t: t.name("T").bound(P.name("U")))), {"U": D.Anything}) is D.Anything
assert PT.DIALECT.infer(typed(P.binop("+", 1, 2))).name() == P.DIALECT.infer(python(P.binop("+", 1, 2))).name() == "int"  # as Python3 infers it

<a id="PYT-03"></a>
## PYT-03 · What a type parameter binds: nothing a term scopes, the statement listing it binding its name; a bound's and a default's names are references

In [ ]:
# A type parameter declares a name that the statement listing it binds (a generic function, class or type alias), not a
# term: none of its kinds refines Let and none has scopes, so a name its bound or its default refers to is a reference
# the scope binds, its own name among them (`T: list[T]`), as mbse-programs' statements bind it.
assert not any(F.is_binder(kinds[tag]) or PT.DIALECT.kinds()[tag].scopes is not None for tag in PT.TAGS)
recursive = typed(lambda u: u.typevar(lambda t: t.name("T").bound(P.subscript(P.name("list"), P.name("T")))))
assert SY.free(recursive, PT.DIALECT) == {"list", "T"}
assert PT.DIALECT.validate(recursive) == ["bound: value: variable 'list' is not bound", "bound: slice: variable 'T' is not bound"]
assert PT.DIALECT.validate(recursive, {"list", "T"}) == []
assert SY.free(typed(lambda u: u.paramspec(lambda t: t.name("P").default_value(P.list_(P.name("A"))))), PT.DIALECT) == {"A"}
assert SY.free(typed(lambda u: u.typevartuple(lambda t: t.name("Ts"))), PT.DIALECT) == set()
# A type expression of Python3's kinds binds as Python3's do: a lambda's parameters within its body, an annotation of
# them outside it, in the scope around the lambda (API 60).
annotated = python(P.lambda_(P.name("x"), (P.arg("x", None, P.name("x")),)))
assert SY.free(annotated, P.DIALECT) == {"x"} and P.DIALECT.kinds()["lambda"].scopes(F.kind_of(annotated)[1]) == [
    frozenset(), {"x"}, frozenset(), frozenset()]  # Let's empty value, its body, its parameter's annotation and default

<a id="PYT-04"></a>
## PYT-04 · A tree mixing type parameters with Python3's expressions, validated and printed from the Types dialect's holder, and a type expression held by a parameter's annotation, from Python3's

In [ ]:
# A tree mixing type parameters with Python3's expressions: a type parameter's bound and default are type expressions,
# Python3's kinds as its Term lists them, holding Python3's terms; validated, inferred and printed from the Types
# dialect's holder.
mixed = lambda u: u.typevar(lambda t: t.name("K").bound(P.subscript(  # noqa: E731
    P.name("Callable"), P.tuple_(P.list_(P.name("int")), optional))).default_value(P.attribute(P.name("typing"), "Any")))
SOURCE = "K: Callable[[int], int | None] = typing.Any"
own = typed(mixed)
assert PTText.ToText(own) == PText.ToText(own) == SOURCE and PTText.ToText is PText.ToText
assert PT.DIALECT.validate(own) == ["bound: value: variable 'Callable' is not bound",
                                   "default_value: value: variable 'typing' is not bound"]
assert PT.DIALECT.validate(own, {"Callable", "typing"}) == [] and SY.free(own, PT.DIALECT) == {"Callable", "typing"}
bound = F.form(own).arguments[0]  # its bound: a term of the Types dialect's union, of Python3's kind
assert F.dialect_of(bound) == PT.DIALECT.union and F.kind_of(bound)[0] == "subscript"
assert F.dialect_of(F.form(bound).arguments[0]) == P.DIALECT.union  # what that holds: Python3's terms
# A parameter's annotation holds the Types dialect's Term: Python3 reads its terms, from Python3's holder. A lambda's
# parameters have no annotation, as Python's grammar has it (mbse-programs' function definitions share the helper),
# so Python3 refuses it there, and its annotation's terms are checked as the Types dialect's.
lam = python(P.lambda_(P.name("x"), (P.arg("x", None, optional),)))
annotation = F.form(lam).arguments[2]  # after Let's empty value and its body: its parameter's annotation, then default
assert F.dialect_of(annotation) == PT.DIALECT.union and PText.ToText(annotation) == "int | None"
assert P.DIALECT.validate(lam) == ["a lambda's parameters have no annotations"]
assert P.DIALECT.validate(python(P.lambda_(P.name("x"), (P.arg("x", None, P.name("T")),)))) == [
    "a lambda's parameters have no annotations", "args args 0 annotation: variable 'T' is not bound"]
typevar_annotation = python(P.lambda_(P.name("x"), (P.arg("x", None, lambda u: u.typevar(lambda t: t.name("1T"))),)))
assert P.DIALECT.validate(typevar_annotation) == ["a lambda's parameters have no annotations",
                                                  "args args 0 annotation: a typevar's name must be an identifier, got '1T'"]

<a id="PYT-05"></a>
## PYT-05 · Invalid syntax where a type expression stands: validated, printed as written, and refused by evaluation

In [ ]:
# Invalid syntax: the Types dialect's Term holds `error` and `missing`, where a type expression stands; validation
# reports them, printers write them as written, and evaluation refuses them by the Types dialect's name.
broken = typed(lambda u: u.typevar(lambda t: t.name("T").bound(lambda b: b.error(lambda e: e.text("int |").terms([
    P.DIALECT.spec(P.name("int"))]))).default_value(lambda d: d.missing(lambda m: m.kind("type")))))
assert PT.DIALECT.validate(broken) == ["bound: invalid syntax 'int |'", "default_value: missing type"]
assert PText.ToText(broken) == "T: int | = "
assert [PT.DIALECT.kind(a)[2].role for a in F.form(broken).arguments] == [F.INVALID, F.INVALID]
with raises(ValueError, match="Python Types error has no text to print"):
    PText.ToText(typed(lambda u: u.paramspec(lambda t: t.name("P").default_value(lambda e: e.error(lambda r: r)))))
with raises(ValueError, match="Python Types error is not evaluated"):
    PV.OfAny(typed(lambda u: u.error(lambda e: e.text("int |"))))
with raises(ValueError, match="Python3 missing is not evaluated"):  # Python3's own, by its name
    PV.OfAny(python(lambda u: u.missing(lambda m: m)))

<a id="PYT-06"></a>
## PYT-06 · Its editions as data: Python 3.0 to 3.15, each with its own kinds it has, every one since Python 3.12

In [ ]:
# Its editions, as data: its define covers Python 3.0 to 3.15, as Python3's, owned by their published names, each with
# its own kinds it has (every one since Python 3.12, as mbse-programs' SINCE has them; a default, 3.13, a feature within
# a kind, not recorded) and none of the kinds it reads; run again, it writes nothing.
types_ = store.Instances.Expressions.Python.Types
VERSIONS = tuple(f"Python 3.{minor}" for minor in range(16))
assert tuple(FS.standards(types_)) == VERSIONS and PT.SINCE == dict.fromkeys(PT.TAGS, 12)
assert FS.tags(FS.find(types_, "Python 3.12")) == ("Python 3.12",) and FS.find(types_, "Python 2.7") is None
assert FS.properties(FS.find(types_, "Python 3.13")) == {"bodies": ("org.python",), "revision": ("version", "3.13")}
assert [len(FS.kinds(edition)) for edition in FS.standards(types_).values()] == [0] * 12 + [3] * 4
assert all(FS.kinds(FS.find(types_, version)) == [kinds[tag] for tag in PT.TAGS] for version in VERSIONS[12:])
assert all(tuple(FS.of(types_, kinds[tag])) == VERSIONS[12:] for tag in PT.TAGS)
assert FS.of(types_, kinds["binop"]) == {} and FS.of(types_, store.registered("Expressions.Framework.Error")) == {}
assert store.transaction(PythonTypes.define)[0]["Python_3_15"] is FS.find(types_, "Python 3.15")
assert tuple(FS.standards(types_)) == VERSIONS  # nothing written again

<a id="PYT-07"></a>
## PYT-07 · Printing every kind as mbse-programs writes type parameters, and a type expression as Python3 writes it

In [ ]:
# Printing, as mbse-programs' printer writes type parameters (`type_var`): a name, its stars, its bound after `:` and its
# default after `=`, each an expression where an element of a display stands (a named expression parenthesized); a type
# expression of Python3's kinds as Python3 writes it.
cases = [
    (lambda u: u.typevar(lambda t: t.name("T")), "T"),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.name("int"))), "T: int"),
    (lambda u: u.typevar(lambda t: t.name("T").default_value(P.name("str"))), "T = str"),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.tuple_(P.name("int"), P.name("str"))).default_value(P.name("int"))),
     "T: (int, str) = int"),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.namedexpr("U", P.name("int")))), "T: (U := int)"),
    (lambda u: u.typevar(lambda t: t.name("T").bound(P.lambda_(P.name("int")))), "T: lambda: int"),
    (lambda u: u.paramspec(lambda t: t.name("P")), "**P"),
    (lambda u: u.paramspec(lambda t: t.name("P").default_value(P.list_(P.name("int"), P.name("str")))), "**P = [int, str]"),
    (lambda u: u.typevartuple(lambda t: t.name("Ts")), "*Ts"),
    (lambda u: u.typevartuple(lambda t: t.name("Ts").default_value(P.starred(P.subscript(
        P.name("tuple"), P.tuple_(P.name("int"), P.constant(...)))))), "*Ts = *tuple[int, ...]"),
    (P.subscript(P.name("dict"), P.tuple_(P.name("str"), optional)), "dict[str, int | None]"),
    (P.constant("Forward"), "'Forward'"),  # a forward reference, a string
    (lambda u: u.paramspec(lambda t: t.default_value(P.name("int"))), "** = int"),  # no name, which validation reports
]
for spec, source in cases:
    assert PText.ToText(typed(spec)) == source, (source, PText.ToText(typed(spec)))
assert {F.kind_of(typed(spec))[0] for spec, _ in cases} >= set(PT.TAGS)  # every kind
assert PTText.TEXTS.keys() == set(PT.TAGS)

<a id="PYT-08"></a>
## PYT-08 · Evaluation refuses each of its kinds by the Types dialect's name; a type expression of Python3's kinds evaluates as Python3's

In [ ]:
# Evaluation (Q11): Python3's evaluator evaluates none of its kinds, and refuses each by the Types dialect's name; a type
# expression of Python3's kinds, which the Types dialect's Term lists, evaluates as Python3 evaluates it.
for spec, tag in [(lambda u: u.typevar(lambda t: t.name("T")), "typevar"), (lambda u: u.paramspec(lambda t: t.name("P")), "paramspec"),
                  (lambda u: u.typevartuple(lambda t: t.name("Ts")), "typevartuple")]:
    with raises(NotImplementedError, match=f"Python Types {tag} is not evaluated"):
        PV.OfAny(typed(spec))
chosen = typed(P.ifexp(P.name("wide"), P.name("int"), P.name("float")))  # `int if wide else float`
assert PV.OfAny(chosen, {"wide": True, "int": "int64", "float": "float32"}) == "int64"
with raises(NotImplementedError, match="Python3 lambda is not evaluated"):  # a lambda first, its annotation not reached
    PV.OfAny(python(P.lambda_(P.name("x"), (P.arg("x", None, lambda u: u.typevar(lambda t: t.name("T"))),))))

<a id="PYT-09"></a>
## PYT-09 · Completeness: each of `ast`'s type parameter nodes, kept as data, is one of its kinds, and each of its kinds one of them

In [ ]:
# Completeness (Q9): `grammars/python3-ast.json` lists `ast`'s type parameter nodes (class `type_param`, Python 3.12 on),
# each mapped to one of the Types dialect's kinds, and every kind is one of them (DIA-20 checks the file whole, and
# against the running Python's `ast`).
grammar = json.loads(Path("../../grammars/python3-ast.json").read_text(encoding="utf-8"))
entries = [node for node in grammar["nodes"] if node["class"] == "type_param"]
assert [(node["node"], node["since"], node["kinds"]) for node in entries] == [
    ("TypeVar", "3.12", ["typevar"]), ("ParamSpec", "3.12", ["paramspec"]), ("TypeVarTuple", "3.12", ["typevartuple"])]
assert {kind for node in entries for kind in node["kinds"]} == set(PT.TAGS)
assert all(int(node["since"].split(".")[1]) == PT.SINCE[kind] for node in entries for kind in node["kinds"])